# Validation score separation

The frozen detector produces different false-positive rates across recordings.
This analysis compares its scores for normal and anomalous observations in
validation groups to investigate those differences. It uses the existing model
and threshold without scoring test recordings. The modeling notebook contains
the earlier exploration.

Install the project and download the training/validation data using the README.
A trusted local artifact from `python -m forge train` is required. Paths below resolve
from the package location, so the repository can live on any drive or in any folder.

In [ ]:
from IPython.display import display

from forge.ml.diagnostics import (
    analyze_validation,
    distribution_figure,
    export_diagnostics,
    histograms,
)

groups, scores, summary, provenance = analyze_validation()
print(f"Model run: {provenance['run_id']}")
print(f"Frozen threshold: {provenance['threshold']:.6f}")
print(f"Partition: {provenance['partition']}")
print(f"Policy: {provenance['policy']}")

## 1. Alert rates and score ranking

FPR is the fraction of normal observations that trigger an alert after the
three-reading persistence rule. Recall is the fraction of anomalous observations
that trigger one. These are point rates, not incident counts.

ROC AUC measures raw score ordering independently of this threshold: 1 is perfect
separation, 0.5 is chance-level ordering, and below 0.5 means the score ordering is
reversed more often than correct. These are descriptive values, without uncertainty
intervals for the temporally dependent observations.

Average precision also depends on class prevalence, included for context. High recall
with high FPR can simply mean almost everything is being flagged.

In [ ]:
display(
    summary[
        [
            "normal_rows",
            "anomaly_rows",
            "fpr",
            "recall",
            "roc_auc",
            "average_precision",
            "anomaly_prevalence",
        ]
    ]
    .sort_values("fpr", ascending=False)
    .round(4)
)

## 2. Normal versus anomalous score distributions

Every panel uses the same score bins and axis scales. Each class is normalized to
100% within its group so a larger class does not dominate the chart. Teal marks
normal annotations; orange marks anomalous annotations. The dashed line is the frozen
threshold. Larger scores mean the detector considers a reading more unusual.

Look for two patterns: both distributions lying above the line indicates many
threshold exceedances; overlapping or reversed distributions indicate a ranking
problem that moving a threshold may not resolve. Histograms describe raw scores,
so their mass above the line is not the persistent-alert FPR shown in the titles.

In [ ]:
binned = histograms(groups, scores)
figure = distribution_figure(binned, summary, provenance)
display(figure)

## 3. Check the picture against numerical summaries

The 10th, 50th (median), and 90th percentiles describe the central score ranges.
`normal_above_threshold` is the raw fraction above the threshold; `fpr` includes
persistence. Persistence can suppress isolated exceedances but cannot fix sustained
high scores throughout normal operation.

In [ ]:
display(
    summary[
        [
            "normal_p10",
            "normal_p50",
            "normal_p90",
            "anomaly_p10",
            "anomaly_p50",
            "anomaly_p90",
            "normal_above_threshold",
            "fpr",
            "anomaly_above_threshold",
            "recall",
        ]
    ].round(4)
)

## 4. Interpretation of the recorded baseline

For run `20261007T003523Z-e78e7b17`, `other/1` has ROC AUC 0.175: its normal-labeled
observations generally rank above its anomalous observations. The shared threshold
also sits below both medians. This is more than a threshold-placement problem.

`other/10` and `valve2/1` have AUC around 0.786 and 0.782. Their anomaly scores
generally rank higher, but normal score medians exceed the shared threshold. A
different operating point may reduce false alerts at a recall cost; that tradeoff
has not been selected or applied here.

`other/7` has AUC 0.996, FPR 0.5%, and recall 99.1%. The failure is therefore not
uniform across the `other` family. Shifts in the score distributions of normal
readings are directly visible. Their physical causes remain unverified.

These findings describe the listed run. If the run printed above differs, interpret
its newly computed table before reusing these numbers. See
[`docs/validation-diagnostics.md`](../docs/validation-diagnostics.md) for the saved
findings and the boundary between diagnosis of model behavior and new model selection.

## 5. Export the figures

The next cell writes a standalone interactive HTML chart, a PNG, a CSV table, and
aggregate JSON under the ignored `reports/figures/validation-<run>/` directory.
It does not change the frozen model or public held-out results. Notebook outputs
are kept out of Git; rerun locally to display the figures.

In [ ]:
output_directory = export_diagnostics(groups, scores, summary, provenance)
print(output_directory)

## 6. Normal operating conditions

Compare the same eight sensor columns across the normal training reference and
normal-labeled observations in validation groups `other/1` and `other/7`.
The training reference includes source-designated normal data as well as normal
annotations; the two validation subsets use only `anomaly == 0`.

The table below uses medians in each sensor's original units. Compare values within each row, since the sensors have different units
and scales. These summaries help
locate differences in normal behavior; they do not establish missing training
coverage, a physical fault, or causal feature attribution for Isolation Forest.

In [ ]:
import pandas as pd
from IPython.display import display

from forge.data.datasets import FEATURES, normal_training

# Reuse the validation-only groups loaded in the first cell.
X_train, training_audit = normal_training()
normal_other1 = groups["other/1"].loc[groups["other/1"]["anomaly"].eq(0), FEATURES]
normal_other7 = groups["other/7"].loc[groups["other/7"]["anomaly"].eq(0), FEATURES]

normal_medians = pd.DataFrame(
    {
        "Training": X_train.median(),
        "Other 1": normal_other1.median(),
        "Other 7": normal_other7.median(),
    }
).rename_axis("Sensor")

normal_counts = pd.Series(
    {"Training": len(X_train), "Other 1": len(normal_other1), "Other 7": len(normal_other7)},
    name="Normal observations",
)

display(normal_counts.to_frame().T)
display(normal_medians.round(4))

### Recorded comparison

The following aggregate table remains visible without running the kernel. It uses
the split associated with model run `20261007T003523Z-e78e7b17`:
18,306 training observations, 557 normal observations in `other/1`, and 743 in `other/7`.

| Sensor (unit) | Training | Other 1 | Other 7 |
|---|---:|---:|---:|
| Accelerometer1RMS (g) | 0.2105 | 0.0804 | 0.2152 |
| Accelerometer2RMS (g) | 0.2649 | 0.1318 | 0.2609 |
| Current (A) | 2.0502 | 1.5977 | 2.5229 |
| Pressure (bar) | 0.0547 | 0.0547 | 0.0547 |
| Temperature (°C, motor) | 88.6761 | 93.8994 | 88.4910 |
| Thermocouple (°C, fluid) | 28.0019 | 22.2418 | 29.7730 |
| Voltage (V) | 229.5140 | 231.0190 | 228.1240 |
| Volume Flow RateRMS (L/min) | 124.3330 | 76.9804 | 127.3200 |

`other/1` has lower median vibration on both accelerometers and lower flow than
the pooled training reference. Its median motor temperature is higher and its
fluid temperature lower. `other/7` is much closer on vibration, flow, and motor
temperature, although its median fluid temperature and current also differ.

Pressure medians are identical across all three sets. Identical medians do not
imply identical distributions or rule out pressure's contribution to the model.

These findings are consistent with different normal operating conditions, but
that remains a hypothesis. A value far from the training median may still fall
inside a well-represented training mode. The forest also responds to combinations
of readings, which this table does not capture. Compare distributions and training
subgroups before deciding how to change the model; these medians alone do not
justify dropping sensors or setting new thresholds.

### 6.1 Compare full distributions

These four panels contain normal observations only. Teal is the pooled normal
training reference; orange and purple are normal validation observations from
`other/1` and `other/7`. Each group's bars sum to 100% within a panel, so the much
larger training set does not dominate by row count. Hover or hide a group through
the legend to inspect overlap. Common bins cover the full observed range without
clipping outliers. Narrow concentrations can appear in only one or two bins.

Look for validation readings beyond the training range, or in gaps between training
concentrations. A range spanning a value does not mean training includes examples
near it. These are marginal plots: they do not show joint sensor combinations or
provide a feature attribution for the forest. The four channels were chosen from
the preceding median review; this is exploratory validation analysis.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from forge.data.sample import SENSOR_UNITS

# Run the preceding Section 6 cell to create these normal-only feature tables.
normal_sets = {
    "Training": X_train,
    "Other 1": normal_other1,
    "Other 7": normal_other7,
}
comparison_sensors = ["Accelerometer1RMS", "Volume Flow RateRMS", "Temperature", "Thermocouple"]
sensor_titles = [
    "Vibration: accelerometer 1",
    "Volume flow rate",
    "Motor temperature",
    "Fluid temperature",
]
group_colors = {"Training": "#267f8c", "Other 1": "#c35432", "Other 7": "#7954a1"}
normal_distributions = make_subplots(
    rows=2, cols=2, subplot_titles=sensor_titles, vertical_spacing=0.19, horizontal_spacing=0.09
)
normal_histogram_rows = []

for index, sensor in enumerate(comparison_sensors):
    row, col = index // 2 + 1, index % 2 + 1
    # Identical bin edges for the three groups within each sensor panel.
    combined = np.concatenate([frame[sensor].to_numpy() for frame in normal_sets.values()])
    edges = np.histogram_bin_edges(combined, bins=80)
    for name, frame in normal_sets.items():
        counts, _ = np.histogram(frame[sensor], bins=edges)
        percentages = 100 * counts / len(frame)
        normal_histogram_rows.extend(
            {
                "sensor": sensor,
                "group": name,
                "left": left,
                "right": right,
                "count": int(count),
                "percent": float(percent),
            }
            for left, right, count, percent in zip(
                edges[:-1], edges[1:], counts, percentages, strict=True
            )
        )
        normal_distributions.add_trace(
            go.Bar(
                x=(edges[:-1] + edges[1:]) / 2,
                y=percentages,
                width=np.diff(edges),
                name=name,
                legendgroup=name,
                showlegend=index == 0,
                marker_color=group_colors[name],
                opacity=0.55,
                hovertemplate="Reading %{x:.4f}<br>%{y:.2f}% of group<extra>%{fullData.name}</extra>",
            ),
            row=row,
            col=col,
        )
    normal_distributions.update_xaxes(
        title_text=f"{sensor} ({SENSOR_UNITS[sensor]})", row=row, col=col
    )
    normal_distributions.update_yaxes(title_text="Normal observations (%)", row=row, col=col)

normal_distributions.update_layout(
    title="Normal operating conditions: training and validation",
    template="plotly_white",
    barmode="overlay",
    height=790,
    legend={"orientation": "h", "y": 1.12},
    margin={"t": 135, "b": 55},
)
display(normal_distributions)

### 6.2 Check coverage using exact observations

The table separates values outside the entire observed training range from values
inside a sparse interval. `Training rows in validation interval` counts training
values between that group's observed minimum and maximum, inclusive. It is a
descriptive count, not a density estimate: wider intervals naturally contain more
training values. No training support in an interval is informative; a positive
count does not establish sufficient coverage or joint support.

The training minimum and maximum describe the observed sample. They are neither
equipment safety limits nor proposed anomaly thresholds. Group intervals are calculated from these
validation observations and are not a deployable operating-mode assignment.

In [ ]:
import pandas as pd

# Calculate support from exact values, not the histogram's bin boundaries.
coverage_rows = []
for sensor in comparison_sensors:
    reference = X_train[sensor]
    for name in ("Other 1", "Other 7"):
        values = normal_sets[name][sensor]
        coverage_rows.append(
            {
                "Sensor": sensor,
                "Group": name,
                "Train minimum": reference.min(),
                "Train maximum": reference.max(),
                "Validation minimum": values.min(),
                "Validation maximum": values.max(),
                "Below training minimum (%)": 100 * values.lt(reference.min()).mean(),
                "Above training maximum (%)": 100 * values.gt(reference.max()).mean(),
                "Training rows in validation interval": int(
                    reference.between(values.min(), values.max(), inclusive="both").sum()
                ),
            }
        )

normal_coverage = pd.DataFrame(coverage_rows).set_index(["Sensor", "Group"])
display(normal_coverage.round(4))

### 6.3 Findings from the recorded comparison

For model run `20261007T003523Z-e78e7b17` and its pinned data split:

- 91.4% of normal `other/1` motor-temperature readings exceed the maximum normal
  training temperature of 91.7249 °C.
- All normal `other/1` fluid-temperature readings (22.2007 to 22.2819 °C) fall
  below the normal training minimum of 24.2667 °C.
- `other/1` vibration and flow lie within the broad training ranges, but no
  training observations fall inside its vibration interval (0.0784 to 0.0836 g)
  or flow interval (75.0401 to 77.9806 L/min).
- Other 7 stays within the training ranges in all four channels. Its vibration
  and flow intervals contain many training observations, although range overlap
  alone does not prove complete coverage.

This establishes gaps in the normal training reference for the observed `other/1`
conditions. It is consistent with high novelty scores even when the source labels
the observations normal. It does not prove which channel causes an individual
Isolation Forest score or establish a physical fault. The forest can flag unfamiliar operation even when the equipment has no
confirmed fault. A maintenance detector needs to distinguish those cases.

No model, feature set, threshold, or held-out result has been changed. The next
experiment needs an explicit normal-coverage strategy using training/development
data; do not simply add these validation rows to training and reuse their validation
score. Any revised final performance claim needs a fresh evaluation plan.